In [ ]:
import os

# Create a .kaggle directory and move kaggle.json there
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json  # Set proper permissions

In [ ]:
!kaggle competitions download -c deepfake-detection-challenge

In [ ]:
import zipfile

with zipfile.ZipFile("/content/deepfake-detection-challenge.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/deepfake")

In [ ]:
import os
import json
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm
from tensorflow.keras.optimizers import Adam

In [ ]:
data_path = "/content/deepfake"
train_video_path = os.path.join(data_path, "train_sample_videos")
test_video_path = os.path.join(data_path, "test_videos")
metadata_path = os.path.join(train_video_path, "metadata.json")

In [ ]:
output_path = "data"  # Output folder

real_dir = os.path.join(output_path, "real")
fake_dir = os.path.join(output_path, "fake")
os.makedirs(real_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)

In [ ]:
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

def extract_frame(video_path, save_path, frame_num=0):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(save_path, frame)
    cap.release()

# Process videos
for video_file, info in tqdm(metadata.items()):
    video_path = os.path.join(train_video_path, video_file)
    if not os.path.exists(video_path):
        print(f"Warning: {video_path} not found!")
        continue

    label = info["label"].lower()
    save_folder = real_dir if label == "real" else fake_dir
    save_path = os.path.join(save_folder, f"{video_file.split('.')[0]}_frame1.jpg")

    extract_frame(video_path, save_path)

print("Frame extraction complete!")

100%|██████████| 400/400 [00:39<00:00, 10.17it/s]

Frame extraction complete!


In [ ]:
# Define parameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
DATA_DIR = "data"  # Base directory containing 'real' and 'fake' subfolders

# Data Augmentation and Loading
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

Found 321 images belonging to 2 classes.
Found 79 images belonging to 2 classes.


In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True  # Freeze base model for transfer learning

# Custom Head for Binary Classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
x = Dense(512, activation="relu")(x)
x = Dense(1, activation="sigmoid")(x)  # Binary classification (Real/Fake)

model = Model(inputs=base_model.input, outputs=x)

# Compile Model
model.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])

# Train Model
history = model.fit(train_generator, validation_data=val_generator, epochs=50)

# Save Model
model.save("deepfake_ResNet50.h5")

print("Training complete and model saved!")

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 88s 3s/step - accuracy: 0.6041 - loss: 1.2754 - val_accuracy: 0.8101 - val_loss: 35.6096
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 859ms/step - accuracy: 0.7829 - loss: 0.8140 - val_accuracy: 0.8101 - val_loss: 6.7443
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 851ms/step - accuracy: 0.8212 - loss: 0.5511 - val_accuracy: 0.8101 - val_loss: 0.4872
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 783ms/step - accuracy: 0.7874 - loss: 0.5322 - val_accuracy: 0.8101 - val_loss: 1.0398
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 821ms/step - accuracy: 0.8188 - loss: 0.4705 - val_accuracy: 0.8101 - val_loss: 0.7577
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 842ms/step - accuracy: 0.8037 - loss: 0.4748 - val_accuracy: 0.8101 - val_loss: 1.4861
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 928ms/step - accuracy: 0.7242 - loss: 0.5609 - val_accuracy: 0.8101 - val_loss: 0.7576
Epoch 8/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 821ms/step - accuracy: 0.8145 - loss: 0.4611 - val_accuracy: 0.

Training complete and model saved!


In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 78,527,365 (299.56 MB)

 Trainable params: 26,158,081 (99.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

 Optimizer params: 52,316,164 (199.57 MB)